In [10]:
import numpy as np
import torch
import torch.nn as nn

graph = np.load("model/graph.npy", allow_pickle=True)
operations = np.load("model/operations.npy", allow_pickle=True)
config = np.load("model/config.npy", allow_pickle=True)
weights = np.load("model/weights.npy", allow_pickle=True)

In [11]:
# Conv2d:
# Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]
# Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists]
def Conv2d(config, weights):
    in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias_flag = config

    layer = nn.Conv2d(
        in_channels=in_ch,
        out_channels=out_ch,
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        bias=bias_flag
    )

    if weights:
        layer.weight.data = weights[0]
        if bias_flag and len(weights) > 1:
            layer.bias.data = weights[1]

    return layer


# BatchNorm2d:
# Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag]
# Weight = [weight(num_features), bias(num_features), running_mean(num_features), running_var(num_features)]
def BatchNorm2d(config, weights):
    num_feat, eps, momentum, affine_flag, track_stats = config

    layer = nn.BatchNorm2d(
        num_features=num_feat,
        eps=eps,
        momentum=momentum,
        affine=affine_flag,
        track_running_stats=track_stats
    )

    if weights:
        layer.weight.data = weights[0]
        layer.bias.data = weights[1]
        layer.running_mean.data = weights[2]
        layer.running_var.data = weights[3]

    return layer


# SiLU:
# Config = [inplace_flag]
# Weight = []
def SiLU(config, weights):
    inplace_flag = config[0]
    return nn.SiLU(inplace=inplace_flag)


# MaxPool2d:
# Config = [kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, ceil_mode_flag]
# Weight = []
def MaxPool2d(config, weights):
    k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode = config

    return nn.MaxPool2d(
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        dilation=(d_h, d_w),
        ceil_mode=ceil_mode
    )


# Upsample:
# Config = [scale_h, scale_w, mode]
# Weight = []
def Upsample(config, weights):
    scale_h, scale_w, mode = config

    return nn.Upsample(
        scale_factor=(scale_h, scale_w),
        mode=mode
    )


# Concat:
# Config = []
# Weight = []
class Concat(nn.Module):
    def __init__(self, dim=1):
        super().__init__()
        self.dim = dim

    def forward(self, inputs):
        return torch.cat(inputs, dim=self.dim)

In [ ]:
def run_graph(graph, operations, config, weights, x):
    """
    x : input tensor
    """
    op_table = {
        "Conv2d": Conv2d,
        "BatchNorm2d": BatchNorm2d,
        "SiLU": SiLU,
        "MaxPool2d": MaxPool2d,
        "Upsample": Upsample,
        "Concat": Concat
    }

    node_outputs = {}

    for node in range(len(operations)):

        op_name = operations[node]
        op_builder = op_table[op_name]

        layer = op_builder(config[node], weights[node])

        inputs = graph[node]

        if inputs == [-1] or inputs == []:
            inp = x
        elif len(inputs) == 1:
            inp = node_outputs[inputs[0]]
        else:
            inp = [node_outputs[i] for i in inputs]

        if op_name == "Concat":
            out = layer(inp)
        else:
            out = layer(inp)

        node_outputs[node] = out

    return node_outputs[len(operations) - 1]